# Wan2.1 VACE 14B の重みを Drive に置く（CPU・1回だけ）

このノートだけが重みを落とします。生成用ノートは Drive を読むだけで、実行中にダウンロードしません。

## 手順

1. メニュー **ランタイム → ランタイムのタイプを変更 → CPU**（ハードウェアアクセラレータは「なし」）にする。GPU のまま実行すると止まります。
2. 上から順にセルを実行する。Drive の接続を求められたら許可する。
3. 終わるまでタブを閉じない。fp16 と fp8 を合わせると大きいです。既に同じ大きさのファイルがあるものは飛ばします。
4. 終わったら最後のセルでランタイムを消す。
5. `minimax-h3-comfyui/refs/sakura-face-crop.jpg` を自分で置く。このノートは顔画像も LoRA も落としません。
6. そのあと生成用ノートを A100 で開く。

置く場所は `マイドライブ/minimax-h3-comfyui/models/` です。


In [ ]:
#@title 1) CPU であることを確認する
import shutil
import subprocess

nvidia = shutil.which("nvidia-smi")
if nvidia:
    result = subprocess.run([nvidia, "-L"], capture_output=True, text=True)
    if result.returncode == 0 and "GPU" in result.stdout.upper():
        raise SystemExit("GPU ランタイムになっている。タイプを CPU に変えて、最初から実行する。")
print("CPU のまま進む")


In [ ]:
#@title 2) Google Drive をマウントする
DRIVE_MODELS = "/content/drive/MyDrive/minimax-h3-comfyui/models"  #@param {type:"string"}
from google.colab import drive

drive.mount("/content/drive")
print(DRIVE_MODELS)


In [ ]:
#@title 3) 公開の重みを Drive に置く
# このノートだけが huggingface.co と GitHub releases から重みを落とす。GPUノートは落とさない。
import subprocess
import sys
from pathlib import Path

BRANCH = "cursor/wan21-vace-colab-a736"
REPO = "fireworker011/Research"
DRIVE_MODELS = "/content/drive/MyDrive/minimax-h3-comfyui/models"
dest = Path("/content/wan_vace/download_manifest.py")
dest.parent.mkdir(parents=True, exist_ok=True)
url = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}/wan_vace/download_manifest.py"
subprocess.run(["wget", "-q", "-O", str(dest), url], check=True)
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
from wan_vace.download_manifest import download_all

download_all(Path(DRIVE_MODELS))
print("Drive に置いた:", DRIVE_MODELS)


ダウンロードが終わってから次のセルを実行すると、この CPU ランタイムが消えます。


In [ ]:
#@title 4) ランタイムを削除する
from google.colab import runtime

runtime.unassign()
